# Create a Custom Operator in C++

DALI allows you to create a custom operator in C++ and load it at runtime. Here are several reasons you might need to write your custom operator:

- DALI does not support the operation that you want to perform and it cannot be expressed by a composition of other operators.
- You want to write an operator that depends on a third-party library.
- You want to optimize your pipeline by providing a manually fused operation in C++.

In this tutorial, we will walk you through the process of writing, compiling, and loading a plugin with a DALI custom operator. For demonstration purposes we will provide a CPU and a GPU implementation for the `CustomDummy` operator. The implementation only copies the input data to the output without any modifications.

For a complete, real-life example, see the [naive_histogram](https://github.com/NVIDIA/DALI/tree/main/docs/examples/custom_operations/custom_operator/naive_histogram) example. It implements a GPU operator that computes a histogram of an image with a CUDA kernel, accepts an argument, and shows how to package the plugin as a Python package.

## Prerequisites

- DALI is installed from the binary distribution or compiled from source.
- The CUDA Toolkit matching the CUDA version of the installed DALI flavor (for example, CUDA 13.x for `nvidia-dali-cuda130`) and a C++20 compiler.
- CMake 3.25.2 or newer.
- You can write in C++.
- You have a basic knowledge of CMake.

<div class="alert alert-warning">

**Important:** A plugin can only be loaded by the exact DALI version and flavor that it was built against. You need to rebuild your plugin every time you install a different DALI version or flavor. See [Building the Plugin](#Building-the-Plugin) for details.

</div>

## Operator Definition

1. Declare the operator in a header file.

2. Provide the common `SetupImpl` function.

The operator class derives from `dali::Operator<Backend>`, where `Backend` is either `dali::CPUBackend` or `dali::GPUBackend`. The implementation of `SetupImpl` can be shared across backends. `SetupImpl` provides the shape and type description of the outputs based on the inputs; the return value determines whether the executor should allocate the storage for the operator's outputs before calling `RunImpl`. `HasContiguousOutputs` declares that the outputs of the operator are contiguous - this is usually true, except for operators which shuffle the batch without copying or legacy operators utilizing `SampleWorkspace` - and that's why we can rely on the default implementation.

In [1]:
from IPython.display import Code

Code(filename="customdummy/dummy.h", language="c++")

#ifndef EXAMPLE_DUMMY_H_
#define EXAMPLE_DUMMY_H_

#include <vector>

#include "dali/pipeline/operator/operator.h"

namespace other_ns {

template <typename Backend>
class Dummy : public ::dali::Operator<Backend> {
 public:
  inline explicit Dummy(const ::dali::OpSpec &spec) :
    ::dali::Operator<Backend>(spec) {}

  virtual inline ~Dummy() = default;

  Dummy(const Dummy&) = delete;
  Dummy& operator=(const Dummy&) = delete;
  Dummy(Dummy&&) = delete;
  Dummy& operator=(Dummy&&) = delete;

 protected:
  bool SetupImpl(std::vector<::dali::OutputDesc> &output_desc,
                 const ::dali::Workspace &ws) override {
    const auto &input = ws.Input<Backend>(0);
    output_desc.resize(1);
    output_desc[0] = {input.shape(), input.type()};
    return true;
  }

  void RunImpl(::dali::Workspace &ws) override;
};

}  // namespace other_ns

#endif  // EXAMPLE_DUMMY_H_

## CPU Operator Implementation

1. Provide the CPU implementation in a C++ implementation file by overriding the `RunImpl` method that accepts a `Workspace`.

2. Register the schema for the custom operator with the `DALI_SCHEMA` macro and register the CPU version of the operator with the `DALI_REGISTER_OPERATOR` macro.

In `RunImpl` we obtain access to the entire batch that is processed. We get the reference to the CPU thread pool from the workspace with `ws.GetThreadPool()` and create tasks (`AddWork`) that will copy samples from input to output in parallel. The second argument of `AddWork` is the priority of the task - here, the size of the sample - so that the tasks are executed from the longest to the shortest to best utilize the worker threads. `RunAll` starts the tasks and waits for their completion.

The outputs are already allocated as we provided the `SetupImpl` function, which returned `true`.

The schema describes the operator: its documentation, inputs, outputs, and arguments (see `AddArg` and `AddOptionalArg`). It is registered once and shared by all the backends of the operator. The name of the schema - `CustomDummy` - determines the name of the operator in Python.

In [2]:
Code(filename="customdummy/dummy.cc", language="c++")

#include "dummy.h"

namespace other_ns {

template <>
void Dummy<::dali::CPUBackend>::RunImpl(::dali::Workspace &ws) {
  const auto &input = ws.Input<::dali::CPUBackend>(0);
  auto &output = ws.Output<::dali::CPUBackend>(0);

  ::dali::TypeInfo type = input.type_info();
  auto &tp = ws.GetThreadPool();
  const auto &in_shape = input.shape();
  for (int sample_id = 0; sample_id < in_shape.num_samples(); sample_id++) {
    tp.AddWork(
        [&, sample_id](int thread_id) {
          type.Copy<::dali::CPUBackend, ::dali::CPUBackend>(
              output.raw_mutable_tensor(sample_id),
              std::nullopt,
              input.raw_tensor(sample_id),
              std::nullopt,
              in_shape.tensor_size(sample_id), 0);
        },
        in_shape.tensor_size(sample_id));
  }
  tp.RunAll();
}

}  // namespace other_ns

DALI_REGISTER_OPERATOR(CustomDummy, ::other_ns::Dummy<::dali::CPUBackend>,
                       ::dali::CPU);

DALI_SCHEMA(CustomDummy)
    .DocStr("Make a c

## GPU Operator Implementation

1. Provide a GPU implementation in a CUDA implementation file by overriding the `RunImpl` method that accepts a `Workspace`.

2. Register the GPU version of the operator with the `DALI_REGISTER_OPERATOR` macro.

As it was the case for the CPU implementation, we obtain the entire batch in the `RunImpl` function. The outputs are already allocated based on the return value of the `SetupImpl` function that was provided earlier.

It is important that we issue the GPU operations on the stream provided by the workspace (`ws.stream()`). Here we copy the batch using `cudaMemcpyAsync`. The `CUDA_CALL` helper function checks the returned error code and throws an exception if the call failed.

In [3]:
Code(filename="customdummy/dummy.cu", language="cuda")

#include <cuda_runtime_api.h>
#include "dummy.h"

namespace other_ns {

template<>
void Dummy<::dali::GPUBackend>::RunImpl(::dali::Workspace &ws) {
  const auto &input = ws.Input<::dali::GPUBackend>(0);
  const auto &shape = input.shape();
  auto &output = ws.Output<::dali::GPUBackend>(0);
  for (int sample_idx = 0; sample_idx < shape.num_samples(); sample_idx++) {
    CUDA_CALL(cudaMemcpyAsync(
            output.raw_mutable_tensor(sample_idx),
            input.raw_tensor(sample_idx),
            shape[sample_idx].num_elements() * input.type_info().size(),
            cudaMemcpyDeviceToDevice,
            ws.stream()));
  }
}

}  // namespace other_ns

DALI_REGISTER_OPERATOR(CustomDummy, ::other_ns::Dummy<::dali::GPUBackend>,
                       ::dali::GPU);

## Building the Plugin

1. Specify the build configuration.

To retrieve the build configuration parameters use `nvidia.dali.sysconfig`.

In [4]:
import nvidia.dali.sysconfig as sysconfig

In [5]:
print(sysconfig.get_include_dir())

/usr/local/lib/python3.10/dist-packages/nvidia/dali/include


In [6]:
print(sysconfig.get_lib_dir())

/usr/local/lib/python3.10/dist-packages/nvidia/dali


In [7]:
print(sysconfig.get_compile_flags())

['-I/usr/local/lib/python3.10/dist-packages/nvidia/dali/include', '-D_GLIBCXX_USE_CXX11_ABI=1']


In [8]:
print(sysconfig.get_link_flags())

['-L/usr/local/lib/python3.10/dist-packages/nvidia/dali', '-ldali']


<div class="alert alert-warning">

**Important:** A plugin is tied to the exact DALI version (and flavor) that it was built against. DALI does not provide ABI compatibility between versions, so:

- A plugin must be linked against the exact same `libdali.so` that is in the DALI Python package directory (`nvidia.dali.sysconfig.get_lib_dir()`) that you intend to use to load your plugin.
- Only one version of `libdali.so` should be loaded in the process at the same time.
- Every time you upgrade (or downgrade) DALI, or switch to a different DALI flavor (for example, from a CUDA 12 to a CUDA 13 build, or to a nightly build), you must rebuild your plugin. Loading a plugin built for a different DALI version can fail with errors such as undefined symbols, or result in undefined behavior.
- Always use the flags returned by `nvidia.dali.sysconfig.get_compile_flags()` (for example, `_GLIBCXX_USE_CXX11_ABI`) when compiling the plugin.

To catch a mismatch early, you can record `nvidia.dali.__version__` and `nvidia.dali.__cuda_version__` (which distinguishes DALI flavors) when building the plugin and compare them with the installed DALI before loading the plugin. The [naive_histogram](https://github.com/NVIDIA/DALI/tree/main/docs/examples/custom_operations/custom_operator/naive_histogram) example shows how to do it.

</div>

2. In this example, we use CMake to build the plugin. The `CMakeLists.txt` file queries `nvidia.dali.sysconfig` for the compile flags and the library directory of the installed DALI.

In [9]:
Code(filename="customdummy/CMakeLists.txt", language="cmake")

cmake_minimum_required(VERSION 3.25.2)
set(CMAKE_CUDA_ARCHITECTURES "80;90")

project(custom_dummy_plugin LANGUAGES CUDA CXX C)

set(CMAKE_CXX_STANDARD 20)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)
set(CMAKE_C_STANDARD 11)

# TODO(klecki): When the test container gets a CMake that supports C++20 as a proper option,
# swap those lines
# set(CMAKE_CUDA_STANDARD 20)
# set(CMAKE_CUDA_STANDARD_REQUIRED ON)
set(CMAKE_CUDA_FLAGS "${CMAKE_CUDA_FLAGS} -std=c++20")

include_directories(SYSTEM "${CMAKE_CUDA_TOOLKIT_INCLUDE_DIRECTORIES}")

execute_process(
        COMMAND python -c "import nvidia.dali as dali; \
                           print(dali.sysconfig.get_lib_dir())"
        OUTPUT_VARIABLE DALI_LIB_DIR
        COMMAND_ERROR_IS_FATAL ANY)
string(STRIP "${DALI_LIB_DIR}" DALI_LIB_DIR)

execute_process(
        COMMAND python -c "import nvidia.dali as dali; print(\" \
                          \".join(dali.sysconfig.get_compile_flags()))"
        OUTPUT_VARIABLE DALI_C

3. We are now ready to compile the plugin that contains the `CustomDummy` custom operator.

In [10]:
! rm -rf customdummy/build
! mkdir -p customdummy/build
# CLICOLOR_FORCE=1 is set by default by Jupyter, but can cause problems with CMake
! cd customdummy/build && \
  CLICOLOR_FORCE=0 cmake .. && \
  make

-- The CUDA compiler identification is NVIDIA 13.0.88 with host compiler GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- The C compiler identification is GNU 11.4.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/lib/python3.10/dist-packages/nvidia/cu13/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Configuring done (3.0s)
-- Generating done (0.0s)
-- Build files have been written to: /opt/dali/docs/examples/custom_operations/custom_operator/

4. After the build is complete we have a dynamic library file that is ready to use.

In [11]:
! ls customdummy/build/*.so

customdummy/build/libdali_customdummy.so


## Importing the Plugin

1. We can see that there is no such operator called `custom_dummy`.

**Note**: Operations available in `nvidia.dali.fn` are automatically converted from camel case to snake case, while the legacy operator objects in `nvidia.dali.ops` keep the camel case format (Example: `fn.custom_dummy` vs. `ops.CustomDummy`).

In [12]:
import nvidia.dali.fn as fn

try:
    help(fn.custom_dummy)
except Exception as e:
    print("Error: " + str(e))

Error: module 'nvidia.dali.fn' has no attribute 'custom_dummy'


2. Load the plugin with `nvidia.dali.plugin_manager.load_library`.

In [13]:
import nvidia.dali.plugin_manager as plugin_manager

plugin_manager.load_library("./customdummy/build/libdali_customdummy.so")

3. Verify that the new operator is available.

In [14]:
help(fn.custom_dummy)

Help on function custom_dummy in module nvidia.dali.fn:

custom_dummy(input, /, *, device=None, name=None, bytes_per_sample_hint=[0], preserve=False)
    Make a copy of the input tensor
    
    Supported backends
     * 'cpu'
     * 'gpu'
    
    
    Args
    ----
    input : TensorList
        Input to the operator.
    
    
    Keyword args
    ------------
    name : str, optional, default = None
        Name of the operator instance. Names must be unique within a pipeline.
        
        Set an explicit name when another API needs to refer to the operator, for
        example through an iterator's ``reader_name`` argument. If omitted, DALI
        generates a name automatically.
    bytes_per_sample_hint : int or list of int, optional, default = `[0]`
        Output size hint, in bytes per sample.
        
        If specified, the operator's outputs residing in GPU or page-locked host memory will be preallocated
        to accommodate a batch of samples of this size.
    pre

For the sake of completeness, it is worth mentioning that even if it is discouraged, it is also possible to access the custom operator through the legacy operator object API (`nvidia.dali.ops.CustomDummy`).

## Using the Operator

The operator can now be used in a pipeline like any other DALI operator. Here we run the CPU implementation and check that the operator returns a copy of its input. To use the GPU implementation, pass GPU data (for example, `fn.custom_dummy(data.gpu())`) and specify a `device_id`.

In [15]:
import numpy as np
from nvidia.dali import pipeline_def


@pipeline_def(batch_size=2, num_threads=2, device_id=None)
def dummy_pipeline():
    data = fn.random.uniform(range=[0, 10], shape=[2, 3], seed=42)
    return data, fn.custom_dummy(data)


pipe = dummy_pipeline()
data, copy = pipe.run()
for sample, copied_sample in zip(data, copy):
    assert np.array_equal(sample, copied_sample)
print(copy.at(0))

[[6.129598  9.241365  7.728935 ]
 [3.6665785 4.298901  6.3396792]]


## A More Complete Operator: `NaiveHistogram`

`CustomDummy` has no arguments and its output has the same shape and type as its input. The [naive_histogram](https://github.com/NVIDIA/DALI/tree/main/docs/examples/custom_operations/custom_operator/naive_histogram) example shows a more typical operator: a GPU operator that computes a histogram of a single-channel image. It takes an argument, and its output has a different shape and type than its input.

1. The constructor reads the value of the `n_bins` argument from the `OpSpec` with `spec.GetArgument<int>("n_bins")`. `SetupImpl` uses it to describe the output: every sample is a 1D tensor of `n_bins` elements of type `DALI_INT32`, regardless of the shape of the input image. Because `SetupImpl` returns `true`, DALI allocates the outputs before calling `RunImpl`.

In [16]:
Code(filename="naive_histogram/naive_histogram.h", language="c++")

// Copyright (c) 2023-2024, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
//
// Licensed under the Apache License, Version 2.0 (the "License");
// you may not use this file except in compliance with the License.
// You may obtain a copy of the License at
//
//     http://www.apache.org/licenses/LICENSE-2.0
//
// Unless required by applicable law or agreed to in writing, software
// distributed under the License is distributed on an "AS IS" BASIS,
// WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
// See the License for the specific language governing permissions and
// limitations under the License.

#ifndef DALI_NAIVE_HISTOGRAM_H
#define DALI_NAIVE_HISTOGRAM_H

#include <vector>
#include "dali/core/tensor_shape.h"
#include "dali/pipeline/operator/operator.h"

namespace naive_histogram {

template<typename Backend>
class NaiveHistogram : public ::dali::Operator<Backend> {
 public:
  explicit NaiveHistogram(const ::dali::OpSpec &spec) :
          ::dali:

2. The schema declares the argument. `AddOptionalArg` declares an argument with a default value (here, `24`); use `AddArg(name, doc, type)` for a required argument. In Python, the arguments are passed as keyword arguments, for example `fn.naive_histogram(images, n_bins=16)`.

   `NumInput(1)` declares a single input. An operator with several inputs declares their number with `NumInput(n)` (or `NumInput(min, max)` for a variable number of inputs) and accesses them with `ws.Input<Backend>(i)`; `ws.NumInput()` returns the number of inputs passed to the operator. The operator is registered only for the `GPU` backend.

In [17]:
Code(filename="naive_histogram/naive_histogram.cc", language="c++")

// Copyright (c) 2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
//
// Licensed under the Apache License, Version 2.0 (the "License");
// you may not use this file except in compliance with the License.
// You may obtain a copy of the License at
//
//     http://www.apache.org/licenses/LICENSE-2.0
//
// Unless required by applicable law or agreed to in writing, software
// distributed under the License is distributed on an "AS IS" BASIS,
// WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
// See the License for the specific language governing permissions and
// limitations under the License.

#include "naive_histogram.h"

namespace naive_histogram {

using namespace ::dali;

DALI_SCHEMA(NaiveHistogram)
                .DocStr("Creates a histogram.")
                .AddOptionalArg("n_bins", "Number of bins in the histogram", 24)
                .NumInput(1)
                .NumOutput(1);

DALI_REGISTER_OPERATOR(NaiveHistogram, NaiveHistogram<GPUBacke

3. `RunImpl` launches a CUDA kernel for every sample of the batch on `ws.stream()`. The output memory allocated by DALI is not initialized, so it is zeroed with `cudaMemsetAsync` before the kernel accumulates the counts with `atomicAdd`. Kernel launches do not return an error code, so the launch is checked with `CUDA_CALL(cudaGetLastError())`.

In [18]:
Code(filename="naive_histogram/naive_histogram.cu", language="cuda")

// Copyright (c) 2023-2026, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
//
// Licensed under the Apache License, Version 2.0 (the "License");
// you may not use this file except in compliance with the License.
// You may obtain a copy of the License at
//
//     http://www.apache.org/licenses/LICENSE-2.0
//
// Unless required by applicable law or agreed to in writing, software
// distributed under the License is distributed on an "AS IS" BASIS,
// WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
// See the License for the specific language governing permissions and
// limitations under the License.

#include "naive_histogram.h"

namespace naive_histogram {

using namespace ::dali;


/**
 * Computes a histogram for a single-channel image.
 *
 * @param input One-channel image.
 * @param input_size Size (in pixels) of the input image.
 * @param n_bins Number of histogram bins.
 * @param histogram Output array. Shall be allocated accordingly to `n_bins`.
 

The [README](https://github.com/NVIDIA/DALI/tree/main/docs/examples/custom_operations/custom_operator/naive_histogram/README.md) of the example describes how to build and run it.

## Preloading the Plugins

DALI can preload plugins automatically via the `DALI_PRELOAD_PLUGINS` environment variable. If provided, the variable is interpreted as a list of paths separated by a colon (`:`). Each of the elements in the list can be either a directory or a library path. Directories are searched recursively, and only files matching the `libdali_*.so` pattern are loaded from them; a library with a different name (for example, `libnaivehistogram.so`) has to be listed by its full path. If the variable is not set, the default plugin directory is scanned, which is the `plugin` directory inside the DALI library installation directory.

Setting `DALI_PRELOAD_PLUGINS` replaces the scan of the default plugin directory: only the listed paths are loaded. The variable is read once, when DALI initializes its operators (on `import nvidia.dali`), so it has to be set before DALI is imported.

For example:

```bash
export DALI_PRELOAD_PLUGINS=/path/to/libdali_customdummy.so:/path/to/plugin/directory
```

## Distributing the Plugin

Because a plugin works only with the DALI version that it was built against, distributing prebuilt plugin binaries is inconvenient. A more convenient approach is to distribute the plugin as a Python source distribution (sdist), which is compiled during the installation against the DALI version that is installed in the target environment. This is how DALI distributes its own TensorFlow and video plugins.

The [naive_histogram](https://github.com/NVIDIA/DALI/tree/main/docs/examples/custom_operations/custom_operator/naive_histogram) example is packaged this way. It uses [scikit-build-core](https://scikit-build-core.readthedocs.io) to drive the CMake build:

In [19]:
Code(filename="naive_histogram/pyproject.toml", language="toml")

[build-system]
requires = ["scikit-build-core>=0.10"]
build-backend = "scikit_build_core.build"

[project]
name = "dali-naive-histogram"
version = "0.1.0"
description = "Example DALI plugin providing the NaiveHistogram operator"
readme = "README.md"
license = { text = "Apache-2.0" }
requires-python = ">=3.10"
# DALI is intentionally not listed in the dependencies: the plugin must be built against the DALI
# already installed in the target environment, so install it with `pip install --no-build-isolation`.
# The name of the DALI distribution depends on its flavor (e.g. `nvidia-dali-cuda120`,
# `nvidia-dali-nightly-cuda130`, or none at all for DALI built from source), so it can't be pinned
# here. Instead, the exact DALI version and flavor are checked when the package is imported.

[tool.scikit-build]
cmake.version = ">=3.25.2"
wheel.packages = ["dali_naive_histogram"]
sdist.exclude = ["build", "dist"]

The `install` rules at the end of `CMakeLists.txt` put the plugin library into the Python package, together with a `_build_info.py` file that records the DALI version and CUDA version (which distinguishes DALI flavors) used for the build:

```cmake
file(WRITE "${CMAKE_CURRENT_BINARY_DIR}/_build_info.py"
     "DALI_VERSION = \"${DALI_VERSION}\"\nDALI_CUDA_VERSION = ${DALI_CUDA_VERSION}\n")
install(TARGETS naivehistogram LIBRARY DESTINATION dali_naive_histogram)
install(FILES "${CMAKE_CURRENT_BINARY_DIR}/_build_info.py" DESTINATION dali_naive_histogram)
```

Importing the package checks that the installed DALI version and flavor match the ones that the plugin was built against and loads the plugin:

In [20]:
Code(
    filename="naive_histogram/dali_naive_histogram/__init__.py",
    language="python",
)

#  Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.

"""Importing this package loads the NaiveHistogram plugin (``fn.naive_histogram``)."""

import os

import nvidia.dali as dali
import nvidia.dali.plugin_manager as plugin_manager

from ._build_info import DALI_CUDA_VERSION, DALI_VERSION

# A plugin can only be loaded by the exact DALI version and flavor it was built against.
# `__version__` does not include the CUDA

To create the source distribution and install it in an environment that already has DALI installed, run:

```bash
cd naive_histogram
pip install build scikit-build-core cmake
python -m build --sdist
pip install --no-build-isolation dist/dali_naive_histogram-0.1.0.tar.gz
```

The `--no-build-isolation` flag is necessary: without it, pip builds the package in an isolated environment that does not contain the DALI installation that the plugin has to be built against. After DALI is upgraded, reinstall the package with `--no-cache-dir --force-reinstall`, so that pip rebuilds the plugin instead of reusing a wheel that it cached for the previous DALI version.

After the installation, `import dali_naive_histogram` makes `fn.naive_histogram` available. Run it from outside of the `naive_histogram` directory: otherwise, Python imports the `dali_naive_histogram` sources from the current directory, which do not contain the built plugin, instead of the installed package.

## Troubleshooting

If loading or running the plugin fails, check the following:

- **`load_library` fails with an undefined symbol, or the process crashes when the plugin is loaded or used.** The plugin was most likely built against a different DALI version or flavor than the one that is installed. Rebuild the plugin after every DALI upgrade, downgrade or flavor switch (for a pip-installed plugin, reinstall it with `--no-cache-dir --force-reinstall`). Make sure that the plugin was built with the include and library directories of the DALI that loads it: compare the paths used in the build with `nvidia.dali.sysconfig.get_include_dir()` and `nvidia.dali.sysconfig.get_lib_dir()` in the environment where the plugin is used, and make sure that only one `libdali.so` is loaded in the process.
- **Undefined symbols that contain `std::__cxx11`.** The plugin was compiled with a different `_GLIBCXX_USE_CXX11_ABI` setting than DALI. Use the flags returned by `nvidia.dali.sysconfig.get_compile_flags()`, which define it.
- **The operator is not available in `nvidia.dali.fn` after loading the plugin.** The name in `fn` is the snake-case version of the name used in `DALI_SCHEMA` (for example, `NaiveHistogram` becomes `fn.naive_histogram`). Check also that `load_library` was called with the plugin that defines the operator, and that the operator is registered for the backend that you use: `NaiveHistogram` works only with GPU inputs.
- **The GPU operator fails with `no kernel image is available for execution on the device`.** The plugin was not compiled for the architecture of your GPU. Set `CMAKE_CUDA_ARCHITECTURES` (or the `CUDAARCHS` environment variable), for example to `native`, and rebuild the plugin.